<img src="http://www.cidaen.es/assets/img/mCIDaeNnb.png" alt="Logo CiDAEN" align="right">




<br><br><br>
<h2><font color="#00586D" size=4>Módulo 8</font></h2>



<h1><font color="#00586D" size=5>Sistema de recomendación de libros</font></h1>

<br><br><br>
<div style="text-align: right">
<font color="#00586D" size=3>Luis de la Ossa</font><br>
<font color="#00586D" size=3>Máster en Ciencia de Datos e Ingeniería de Datos en la Nube</font><br>
<font color="#00586D" size=3>Universidad de Castilla-La Mancha</font>

</div>

---

<a id="indice"></a>
<h2><font color="#00586D" size=5>Índice</font></h2>


* [1. Introducción](#section1)
   * [Lectura de las revisiones](#section11)
   * [Etiquetas](#section12)
   * [Selección de cinco libros para pruebas](#section13)
* [2. Búsqueda por similaridad: tf-idf](#section2)   
    * [Limpieza y preparación del texto](#section21)
    * [Búsqueda por similaridad](#section22)  
* [3. Búsqueda por similaridad: embeddings](#section3)
* [4. Recomendación basada en contenido](#section4)
   * [Selección de libros preferidos por el usuario](#section41)
   * [Búsqueda de libros similares](#section42)
   * [Priorización de resultados y recomendación](#section43)
* [5. Sistema híbrido de recomendación](#section5)  
<br>

In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

import warnings
warnings.filterwarnings('ignore')

<a id="section1"></a>
## <font color="#00586D"> 1. Introducción</font>
<br>

Los sistemas de recomendación basados en filtrado colaborativo utilizan de manera exclusiva los perfiles de votación de usuarios/items, y no consideran el contenido. Como se vio en clase,  permiten obtener resultados que, si bien cualitativamente son aceptables, pueden llegar a desconcertar al no corresponderse con lo esperado. 

En este proyecto se diseñará inicialmente un pequeño sistema de recomendación de libros basado en contenido. Para ello, será necesario hacer uso de algunos de los conceptos relaccionados con aprendizaje automático sobre información textual (_Text Mining_) que han sido tratados también a lo largo del módulo. También se implementará una alternativa basada en *embeddings*, más acorde con las tecnologías que se están utilizando desde hace unos meses. Finalmente, se desarrollará un sistema híbrido que se valdrá de filtrado colaborativo para hacer la recomendación a partir de una preselección de libros.

Como punto de partida, se partirá del conjunto de datos [goodbooks-10k](https://www.kaggle.com/zygmunt/goodbooks-10k) disponible en [kaggle](https://www.kaggle.com). Éste contiene información relativa a 10000 libros obtenida de la red social  [goodreads](http://goodreads.com), que actualmente es el sitio de referencia en la red para aficcionados a la lectura. Además de títulos y autores, el conjunto de datos incluye votos y etiquetas aportadas por más de 53000 usuarios. 

<div class="alert alert-block alert-warning">
<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
Se han hecho algunas modificaciones con respecto a la base original para que sea menos tedioso manejar los distintos índices e identificadores. 
</div>

In [3]:
import pandas as pd
import numpy as np

df_goodreads = pd.read_csv('data/books.csv', sep="\t")
df_goodreads.head(2)

,gr_book_id,gr_best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,title,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,"The Hunger Games (The Hunger Games, #1)",...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,Harry Potter and the Sorcerer's Stone (Harry P...,...,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...


<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>

---

<a id="section11"></a> 
###  <font color="#00586D"> Lectura de las revisiones </font>
<br>

Los datos de la tabla apenas contienen información relativa al contenido de cada libro. Sin embargo, es posible acceder a los resúmenes almacenados en la propia web [GoodReads](www.goodreads.com) que, en cierto grado, aportan esta información.  El resumen de cada libro ha sido almacenado en un archivo de texto denominado `./data/overviews/gr_book_id.txt`, donde `gr_book_id` corresponde al identificador del libro (columna `gr_book_id` de `df_goodreads`). 

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Implementar una función, denominada `get_overview(gr_book_id)`, que reciba el identificador de un libro, lea el archivo de texto que contiene el resumen correspondiente y lo devuelva en un _String_ (o devuelva `None` si este resumen no existe o tiene tamaño 0). 

In [ ]:
def get_overview(gr_book_id):
    import os
    path = f'./data/overviews/{gr_book_id}.txt'
    if not os.path.isfile(path):
        return None
    with open(path, 'r', encoding='utf-8') as f:
        overview = f.read().strip()
    return overview if overview else None

    
get_overview(320) 

'(Book Jacket Status: Jacketed)The brilliant, bestselling, landmark novel that tells the story of the Buendia family, and chronicles the irreconcilable conflict between the desire for solitude and the need for love—in rich, imaginative prose that has come to define an entire genre known as "magical realism."'

<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true"></i></font> Crear una columna en `df_goodreads`, denominada `overview`, que contenga la revisión del libro correspondiente. Rellenar los valores vacíos de esa columna con un _String_ de longitud cero ("").

In [5]:
df_goodreads['overview'] = df_goodreads['gr_book_id'].apply(lambda x: get_overview(x) or "")


<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

Por mejorar visualizar de forma más clara, se preservarán solamente las columnas relevantes, y se establecerá la columna `gr_book_id` como índice.

In [6]:
columns = ['gr_book_id','authors', 'original_publication_year', 'title', 'average_rating','overview']
df_goodreads = df_goodreads[columns].set_index('gr_book_id')
df_goodreads.head(2)

,authors,original_publication_year,title,average_rating,overview
gr_book_id,,,,,
2767052,Suzanne Collins,2008.0,"The Hunger Games (The Hunger Games, #1)",4.34,Winning will make you famous. Losing means cer...
3,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Sorcerer's Stone (Harry P...,4.44,Harry Potter's life is miserable. His parents ...


<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true"></i></font> Eliminar todas las filas en las que la columna `overview` no tiene contenido. 

In [7]:
df_goodreads = df_goodreads[df_goodreads['overview'] != ""]


<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>

---

<a id="section12"></a> 
###  <font color="#00586D">Etiquetas </font>
<br>


En el conjunto de datos se proporcionan etiquetas relativas a cada libro que han sido aportadas por los usuarios. Esta información está incluída en dos archivos. El primero de ellos, `tags.csv`, contiene el identificador de cada etiqueta y el código correspondiente. El segundo, `book_tags.csv` contiene las etiquetas relativas a cada libro. Se almacenarán, respectivamente, en los _DataFrame_ `df_tags` y `df_book_tags`.

In [8]:
df_tags = pd.read_csv('./data/tags.csv')
df_book_tags = pd.read_csv('./data/book_tags.csv')

print('Etiquetas', end='')
display(df_tags.iloc[2000:2002])

print('\n Etiquetas por libro', end='')
display(df_book_tags.head(3))

Etiquetas

,tag_id,tag_name
2000,2000,alex-read
2001,2001,alex-rider



 Etiquetas por libro

,gr_book_id,tag_id,count
0,1,30574,167697
1,1,11305,37174
2,1,11557,34173


A continuación se funden los dos conjuntos de datos en uno, denominado `df_book_tags`.

In [9]:
df_book_tags=df_book_tags.merge(df_tags, left_on='tag_id', right_on="tag_id").drop(columns=['count', 'tag_id'])
df_book_tags.tail()

,gr_book_id,tag_name
999907,33288638,neighbors
999908,33288638,kindleunlimited
999909,33288638,5-star-reads
999910,33288638,fave-author
999911,33288638,slowburn


<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Inspeccionar las etiquetas que aparecen en `df_book_tags` y el número de veces que aparece cada una. ¿Deberían eliminarse algunas?

In [10]:
etiquetas_conteo = df_book_tags['tag_name'].value_counts()
etiquetas_conteo


tag_name
to-read               9983
favorites             9881
owned                 9858
books-i-own           9799
currently-reading     9776
                      ... 
prose-literature         1
best-reads-2016          1
on-the-stack             1
beautiful-pictures       1
slowburn                 1
Name: count, Length: 34252, dtype: int64

<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

Una de las cosas que se observan es que hay un alto número de etiquetas que aparecen una o muy pocas veces, y que son irrelevantes, por lo que es mejor ignorarlas. 


<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Eliminar todas las etiquetas que aparezcan menos de 20 veces. 

<div class="alert alert-block alert-warning">
    
<i class="fa fa-exclamation-circle" aria-hidden="true"></i> Este ejercicio se puede hacer de varias formas. Una de ellas, consiste en agrupar `df_book_tags` por `tag_name`, mediante `groupby`, y filtrar con `filter` los grupos con tamaño >=20.
</div>

In [12]:
df_book_tags = df_book_tags.groupby('tag_name').filter(lambda x: len(x) >= 20)



<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

Otro de los problemas que se aprecian es que algunas etiquetas son genéricas, y no corresponden a libros concretos. Por ejemplo palabras como `read-readings` o `favourites`. 

In [ ]:
# COMPLETAR

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Eliminar todas las etiquetas que contengan estos términos (los términos de la lista `target_tags`).  

<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
En este caso, es recomendable no utilizar las etiquetas completas para eliminar así sus variantes también. Por otra parte, `Series.str.contains` acepta expresiones regulares.
</div>

In [13]:
target_tags = ['read', 'favo', 'own', 'top', 'book', 'librar', 'kindle', 'list', 'to_buy', 'current']
pattern = '|'.join(target_tags)

df_book_tags = df_book_tags[~df_book_tags['tag_name'].str.contains(pattern, case=False, regex=True)]

    
df_book_tags

,gr_book_id,tag_name
1,1,fantasy
4,1,young-adult
5,1,fiction
6,1,harry-potter
9,1,ya
...,...,...
999899,33288638,reviewed
999902,33288638,stand-alones
999903,33288638,alpha
999904,33288638,kick-ass-heroine


<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

Por último,  se crea un `DataFrame` denominado `df_book_tag_text` en el que, para cada libro (indicado por su código `goodreads_book_id`), se añade una columna con _un solo campo de texto_, resultado de unir las etiquetas correspondientes.  Para ello, se agrupan las entradas `DataFrame` en `df_book_tags` en función del campo `goodreads_book_id` y se unen todas las etiquetas de cada grupo mediante `join`.

In [14]:
df_book_tag_text = (df_book_tags.groupby('gr_book_id')['tag_name']
                                .apply(lambda group_tags: f"{' '.join(group_tags):s}")
                                .to_frame()
                                .rename(columns={'tag_name':'tags'}))
        
df_book_tag_text.head()

,tags
gr_book_id,
1,fantasy young-adult fiction harry-potter ya se...
2,fantasy children children-s default fiction yo...
3,fantasy young-adult fiction harry-potter ya se...
5,fantasy young-adult fiction harry-potter ya se...
6,fantasy young-adult fiction harry-potter ya se...


<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Incorporar la información del texto de las etiquetas, es decir, la columna `tags`, al `DataFrame` principal `df_goodreads` (utilizar _merge_).

In [21]:
df_goodreads = df_goodreads.merge(df_book_tag_text, left_index=True, right_index=True, how='left')


MergeError: Passing 'suffixes' which cause duplicate columns {'tags_x'} is not allowed.

Sin querer volvi a ejecutar la celda.

<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

In [27]:
df_goodreads = df_goodreads.drop(columns=['tags_y'])  


In [28]:
df_goodreads.head()

,authors,original_publication_year,title,average_rating,overview,tags
gr_book_id,,,,,,
2767052,Suzanne Collins,2008.0,"The Hunger Games (The Hunger Games, #1)",4.34,Winning will make you famous. Losing means cer...,young-adult fiction fantasy ya science-fiction...
3,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Sorcerer's Stone (Harry P...,4.44,Harry Potter's life is miserable. His parents ...,fantasy young-adult fiction harry-potter ya se...
41865,Stephenie Meyer,2005.0,"Twilight (Twilight, #1)",3.57,About three things I was absolutely positive.F...,young-adult fantasy vampires ya fiction parano...
2657,Harper Lee,1960.0,To Kill a Mockingbird,4.25,The unforgettable novel of a childhood in a sl...,classics classic historical-fiction school clà...
4671,F. Scott Fitzgerald,1925.0,The Great Gatsby,3.89,"On its first publication in 1925, The Great Ga...",classics fiction classic literature school his...


<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>


---

<a id="section2"></a> 
## <font color="#00586D"> 2. Búsqueda por similaridad:  tf-idf </font>
<br>

La información _tf-idf_ es muy útil de cara a clasificar y comparar documentos. Cada texto se puede representar  mediante un vector de valores _tf-idf_, y la búsqueda de documentos se apoya en una medida de similaridad, la _similaridad coseno_, para esta representación.


<a id="section21"></a> 
###  <font color="#00586D">Limpieza y preparación del texto </font>
<br>

En este caso, el contenido (texto) asociado a cada libro se representa mediante un modelo de bolsa de palabras. Como se ha explicado a lo largo del módulo, cuando se trabaja con este tipo de representación es recomendable limpiar el texto y eliminar información irrelevante. Para ello se utilizará la librería `spacy`.  En primer lugar, se utilizará el modelo de lenguaje `en_core_web_sm`. Aunque no es el modelo con mejor rendimiento, es suficiente para este contexto, y es más eficiente que `en_core_web_trf`(basado en *transformers*, y que es el que mejor funciona).

In [ ]:
#! python -m spacy download es_dep_news_trf
#! python -m spacy download en_core_web_sm # Solo se utilizará este
#! python -m spacy download en_core_web_trf 

In [29]:
import spacy 

# Crea un objeto con el pipeline
nlp = spacy.load("en_core_web_sm")
nlp.pipe_names

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Crear una función denominada `clean` que acepte un texto, lo convierta en un documento (`spacy`) y devuelva un *String* compuesto por los lemas correspondientes a cada token, en minúscula, y descartando los tokens que no sean alfanuméricos, o los que correspondan a *stopwords*.

In [30]:
def clean(overview):
    doc = nlp(overview)
    tokens = [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha and not token.is_stop
    ]
    return " ".join(tokens)


overview = df_goodreads.iloc[0]['overview']
clean(overview)

'win famous lose mean certain death nation panem form post apocalyptic north america country consist wealthy capitol region surround poor district early history rebellion lead district capitol result destruction creation annual televise event know hunger games punishment reminder power grace capitol district yield boy girl age lottery system participate game tribute choose annual reaping force fight death leave survivor claim victory year old katniss young sister prim select district female representative katniss volunteer place male counterpart peeta pit big strong representative train life see death sentence katniss close death survival second nature'

<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Crear una serie denominada `text_books` concatenando el resultado de aplicar la función `clean` sobre cada elemento de `overview`, y la columna `tags` (con un espacio entre medias).


<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
Este ejercicio tarda unos cinco minutos en ejecutarse. Se ha planteado así (sin utilizar `nlp.pipe`) porque es más sencillo, y en este caso tarda prácticamente lo mismo. Por otra parte, como el resultado es de uso puntual para este apartado, no se añade al `DataFrame` y se almacena en una serie aparte.
</div>

In [31]:
text_books = df_goodreads.apply(
    lambda row: f"{clean(row['overview'])} {row['tags']}", axis=1
)


<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>

---

<a id="section22"></a> 
###  <font color="#00586D">Búsqueda por similaridad</font>
<br>


<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i></font> Crear un objeto `TfidfVectorizer` de `sklearn`, denominado `tfidf_vect`, que represente un máximo de 20000 términos. Obtener la matriz _tf-idf_ de los datos almacenados en `text_books` y almacenarla en una variable denominada `text_books_tfidf`. Extraer los términos considerados en `tfidf_vect`, y almacenarlos en la variable `terms`.

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer(max_features=20000)
text_books_tfidf = tfidf_vect.fit_transform(text_books)
terms = tfidf_vect.get_feature_names_out()

<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

Se tomará un libro incluído en el conjunto de datos `df_goodreads`. Por ejemplo, *"Crepúsculo"*.

In [33]:
twilight_id = 41865
twilight_pos = df_goodreads.index.get_loc(twilight_id)

print(df_goodreads.loc[twilight_id])
text_query = df_goodreads.loc[twilight_id]['overview']
text_query

authors                                                        Stephenie Meyer
original_publication_year                                               2005.0
title                                                  Twilight (Twilight, #1)
average_rating                                                            3.57
overview                     About three things I was absolutely positive.F...
tags                         young-adult fantasy vampires ya fiction parano...
Name: 41865, dtype: object


"About three things I was absolutely positive.First, Edward was a vampire.Second, there was a part of him—and I didn't know how dominant that part might be—that thirsted for my blood.And third, I was unconditionally and irrevocably in love with him.In the first book of the Twilight Saga, internationally bestselling author Stephenie Meyer introduces Bella Swan and Edward Cullen, a pair of star-crossed lovers whose forbidden relationship ripens against the backdrop of small-town suspicion and a mysterious coven of vampires. This is a love story with bite."

Para poder buscar otro libro por similaridad, es necesario transformar el texto de consulta a la representación _tf-idf_. Previamente, ha de ser preprocesado (limpiado con la función `clean`).

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Transformar el documento `text_query` a formato _tf-idf_ y almacenar el resultado en `text_query_tfidf`.

<div class="alert alert-block alert-warning">
    
<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
Esta sería la manera de proceder con libros nuevos. No obstante, en este caso concreto, ___como el libro ya estaba en el corpus y se ha obtenido su representación___, se podría obtener el vector directamente desde `text_books_tfidf`.

</div>

In [ ]:
text_query_cleaned = clean(text_query)

text_query_tfidf = tfidf_vect.transform([text_query_cleaned])


<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Mostrar los 10 términos más relevantes (con mayor valor _tf-idf_) para el documento `text_query`.

In [ ]:
import numpy as np

tfidf_array = text_query_tfidf.toarray().flatten()

top_indices = np.argsort(tfidf_array)[::-1][:10]

top_terms = [(terms[i], tfidf_array[i]) for i in top_indices]

for term, value in top_terms:
    print(f"{term}: {value:.4f}")


edward: 0.3174
unconditionally: 0.2304
ripen: 0.2304
stephenie: 0.2092
vampire: 0.2060
coven: 0.2027
cullen: 0.2009
dominant: 0.1918
meyer: 0.1918
bella: 0.1894


<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Calcular las similaridades (similaridad coseno) del documento `text_query_tfidf` con el resto de documentos en la matriz `text_books_tfidf`, y almacenarlas en un vector denominado `similarities_tfidf`.

<div class="alert alert-block alert-warning">
    
<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
La función `cosine_similarity` toma como argumentos dos matrices y devuelve una matriz (aunque sea unidimensional). Para obtener un vector unidemensional, y operar posteriormente con más agilidad, `similarities_tfidf` ha de almacenar el elemento cero de la matriz devuelta por la función, o transformar esta matriz en un vector. 
</div>

In [36]:
from sklearn.metrics.pairwise import cosine_similarity

similarities_matrix = cosine_similarity(text_query_tfidf, text_books_tfidf)

similarities_tfidf = similarities_matrix[0]

print(similarities_tfidf[:10])


[0.01781354 0.00527974 0.72253601 0.02319492 0.01892415 0.02201402
 0.         0.01358402 0.00981124 0.00784296]


<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Obtener los índices de los 6 libros más similares a `text_query` (obviamente, el más similar será el propio libro).

<div class="alert alert-block alert-warning">
    
<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
Hay que tener en cuenta que los índices de los libros no corresponden a su posición. En este caso hay que acceder por posición, ya que `similarities_tfidf` es un array `Numpy`.

</div>

In [37]:
top_similarities = np.argsort(similarities_tfidf)[::-1][:6]

df_goodreads.iloc[top_similarities]['title']


gr_book_id
41865                                Twilight (Twilight, #1)
3090465                   The Twilight Saga (Twilight, #1-4)
6441654    Twilight and Philosophy: Vampires, Vegetarians...
7140220                                 Twilight and History
49041                                New Moon (Twilight, #2)
1162543                         Breaking Dawn (Twilight, #4)
Name: title, dtype: object

<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

Se aprecia que, efectivamente, los libros son muy similares. De hecho, todos corresponden a la saga. Esto puede deverse al uso de palabras clave muy concretas como _"Twilight"_. 


<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Repetir la prueba con el libro con `gr_book_id` `7613`,  *"Animal Farm"*.

In [38]:
animalfarm_id = 7613

overview = df_goodreads.loc[animalfarm_id]['overview']
tags = df_goodreads.loc[animalfarm_id]['tags']

text_query = clean(overview) + ' ' + tags

text_query_tfidf = tfidf_vect.transform([text_query])

similarities_tfidf = cosine_similarity(text_query_tfidf, text_books_tfidf)[0]

top_similarities = np.argsort(similarities_tfidf)[::-1][:6]

df_goodreads.iloc[top_similarities]['title']


gr_book_id
7613                    Animal Farm
5472             Animal Farm / 1984
7624              Lord of the Flies
32650    The Return of the Native  
3102                    Howards End
5344                     Hard Times
Name: title, dtype: object

<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

Aunque algunos de los libros recomendados tienen cierta relación, otros evidentemente no.

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>


---

<a id="section3"></a> 
## <font color="#00586D"> 3. Búsqueda por similaridad:  embeddings </font>
<br>


Con la llegada de nuevos modelos de *Deep Learning*, como los *Transformers*, los modelos de lenguaje han aumentado sus capacidades hasta un punto que era impensable hace cinco años. A un nivel algo más doméstico, los *embeddings* que antes codificaban palabras y relaciones entre ellas, ahora son capaces de codificar textos de cierta extensión.

En este apartado, se utilizarán *embeddings* para almacenar los resúmenes de los libros y buscar por similaridad. Además, se utilizará *ChromaDB*, una base de datos específica para este tipo de representación que, entre otras cosas, permite automatizar la representación o llevar a cabo las búsquedas por similaridad.  

<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i> Por defecto, ChromaDB utiliza el modelo `all-MiniLM-L6-v2` para calcular los embeddings. Por algún motivo,  utilizar la función de creación de *embeddings* por defecto puede dar problemas en algunos equipos (Mac con procesador Intel) o ser extremadamente lenta (Mac con procesador Silicon). Mediante el parámetro `embedding_function` se puede especificar un modelo concreto al crear o recuperar una colección. Incluso si el modelo pasado sigue siendo `all-MiniLM-L6-v2`, parece solucionarse el problema. 
</div>

In [39]:
#!pip install chromadb
#!pip install sentence_transformers

from chromadb.utils import embedding_functions

# Crea la función para calcular embeddings que puede ser pasada como parámetro
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
#emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L12-v2") # Posterior

Aunque Chroma DB proporciona un servicio de almacenamiento, se puede utilizar en modo local. Además, permite crear un cliente persintente que evita tener que crear los *embeddings* en cada sesión (lleva unos 8-10 minutos o más, dependiendo del equipo y de la función de embedding). En la siguiente celda se crea el cliente y se trata de acceder a la colección `libros` que contiene los *embeddings*. Si no existe, la crea y añade todos los documentos. 

In [40]:
import chromadb
# Así se crearía un cliente no persistente
#client = chromadb.Client()

# Crea el cliente en la carpeta books_ebd
client = chromadb.PersistentClient(path="books_ebd")

# Intenta acceder a la colección especificando la función que 
# se utiliza para crear los embeddings. Si no puede, la crea.
try:
    #collection = client.get_collection("libros")    
    collection = client.get_collection("libros", embedding_function = emb_fn)
except:
    #collection = client.create_collection("libros")  
    collection = client.create_collection("libros", embedding_function = emb_fn)    
    # añade todos los documentos.
    for identifier, row in df_goodreads.iterrows():
        collection.add(
            documents = row['overview'],         # Textos
            ids = str(identifier),               # Identificador (debe ser un Strng)              
            metadatas={"title": row['title'],    # Campos
                       "author": row['authors']} 
        )   

La siguiente celda obtiene los 10 libros más similares a "Crepúsculo".

In [41]:
results = collection.query(
    query_texts=df_goodreads.loc[twilight_id]['overview'],
    n_results=10
)

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Explorar la estructura `results`. Mostrar los títulos y autores de los diez resultados.


In [46]:
similar_ids = [int(i) for i in results['ids'][0]]

df_goodreads.loc[similar_ids, ['title', 'authors']]



,title,authors
gr_book_id,,
41865,"Twilight (Twilight, #1)",Stephenie Meyer
49041,"New Moon (Twilight, #2)",Stephenie Meyer
6441654,"Twilight and Philosophy: Vampires, Vegetarians...","Rebecca Housel, J. Jeremy Wisnewski, William I..."
11450360,"Twilight: The Graphic Novel, Vol. 2 (Twilight...","Stephenie Meyer, Young Kim"
7619292,"Twilight: The Graphic Novel, Vol. 1 (Twilight:...","Young Kim, Stephenie Meyer"
7140220,Twilight and History,Nancy Reagin
690926,"The Twilight Collection (Twilight, #1-3)",Stephenie Meyer
8726744,The Twilight Saga Complete Collection (Twilig...,Stephenie Meyer
3090465,"The Twilight Saga (Twilight, #1-4)","Stephenie Meyer, Ilyana Kadushin, Matt Walters"


<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Obtener y mostrar los 10 libros más similares a *"Animal Farm"* (id `animalfarm_id`).


In [47]:
results = collection.query(
    query_texts=df_goodreads.loc[animalfarm_id]['overview'],
    n_results=10
)

similar_ids = [int(i) for i in results['ids'][0]]

df_goodreads.loc[similar_ids, ['title', 'authors']]



,title,authors
gr_book_id,,
7613,Animal Farm,George Orwell
5472,Animal Farm / 1984,"George Orwell, Christopher Hitchens"
66933,The Wretched of the Earth,"Frantz Fanon, Jean-Paul Sartre, Richard Philco..."
105703,Pride of Baghdad,"Brian K. Vaughan, Niko Henrichon"
428,Play It as It Lays,"Joan Didion, David Thomson"
11107244,The Better Angels of Our Nature: Why Violence ...,Steven Pinker
93269,Earth Abides,George R. Stewart
5805,V for Vendetta,"Alan Moore, David Lloyd"
29340182,Shrill: Notes from a Loud Woman,Lindy West


<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

Se aprecia ahora que los libros son más parecidos en la temática.

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>


---

<a id="section4"></a> 
## <font color="#00586D"> 4. Recomendación basada en contenido </font>
<br>

Existen muchas posibilidades en la implementación de sistemas de recomendación basados en contenido. En general, todas requieren:

* Determinar qué ítems prefiere el usuario
* Encontrar ítems similares
* Priorizar esos ítems


Para ilustrar el funcionamiento de este tipo de sistemas, se tomarán los datos relativos a la actividad de un usuario escogido al azar, y se devolverá un conjunto de libros recomendados. 


El archivo `'./data/ratings.csv'` contiene las valoraciones hechas por más de 53000 usuarios a los 10000 libros. En total, contiene cerca de un millón de entradas en formato (`user_id`, `gr_book_id`, `rating`). El tamaño de la base de datos dificulta el trabajo con una matriz, aunque sea dispersa, ya que el proceso de elaboración (mediante `pivot` es muy lento). 

In [48]:
df_ratings = pd.read_csv('./data/ratings.csv', sep='\t')
display(df_ratings.head())
df_ratings.shape

,user_id,gr_book_id,rating
0,313,2767052,5
1,438,2767052,3
2,587,2767052,5
3,1168,2767052,4
4,1184,2767052,4


(981756, 3)

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>


---

<a id="section41"></a> 
### <font color="#00586D"> Selección de libros preferidos por el usuario </font>
<br>

El primer paso de la recomendación consiste en determinar qué libros prefiere el usuario.  Una posibilidad consiste en seleccionar aquellos para los que éste ha otorgado una puntuación positiva. 

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Tomar un usuario al azar y devolver las entradas correspondientes a los libros que ha votado. Almacenarlas en el `DataFrame` `ratings_user`. Descartar de `ratings_user` todos los libros con puntuación menor que cuatro. 

In [50]:
np.random.seed(0)
test_user = np.random.randint(max(df_ratings['user_id']))

ratings_user = df_ratings[df_ratings['user_id'] == test_user]
ratings_user = ratings_user[ratings_user['rating'] >= 4]
ratings_user

,user_id,gr_book_id,rating
53101,2732,59980,5
209776,2732,19030845,5
214080,2732,16992,5
242344,2732,59952,4
369792,2732,11734,4
401262,2732,16982,4
425738,2732,101911,5
476351,2732,543339,4
524632,2732,97898,5
543609,2732,141372,4


<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>


---

<a id="section42"></a> 
### <font color="#00586D"> Búsqueda de libros similares</font>
<br>


 

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Almacenar los `gr_book_id` de los libros valorados positivamente por el usuario (`ratings_user`) en una serie denominada `books_user`. Mostrar esos libros (seleccionar las filas correspondientes en `df_goodreads`).

In [51]:
books_user = ratings_user['gr_book_id']
df_goodreads.loc[books_user]

,authors,original_publication_year,title,average_rating,overview,tags
gr_book_id,,,,,,
59980,"Frank Miller, David Mazzucchelli, Richmond Lew...",1987.0,Batman: Year One,4.23,Lieutenant James Gordon takes up a new post in...,cómics graphic-novel batman fiction comics-gra...
19030845,Frank Miller,1986.0,Batman: The Dark Knight Returns #1,4.21,This masterpiece of comics storytelling brings...,comics graphic-novels cómics graphic-novel bat...
16992,"Mark Waid, Alex Ross, Elliot S. Maggin",1996.0,Kingdom Come,4.24,"Writer Mark Waid, coming from his popular work...",comics graphic-novels graphic-novel dc còmics ...
59952,"Frank Miller, Lynn Varley",1998.0,300,3.93,The army of Persia - a force so vast it shakes...,graphic-novels comics cómics graphic-novel his...
11734,Philip Roth,2000.0,"The Human Stain (The American Trilogy, #3)",3.84,"It is 1998, the year in which America is whipp...",fiction novels american contemporary literatur...
16982,"Kurt Busiek, Alex Ross",1993.0,Marvels,4.21,"Welcome to New York. Here, burning figures roa...",comics graphic-novels graphic-novel marvel fic...
101911,A. Manette Ansay,1994.0,Vinegar Hill,3.36,"In a stark, troubling, yet ultimately triumpha...",fiction oprah contemporary-fiction contemporar...
543339,Michael Blake,1988.0,"Dances with Wolves (Dances with Wolves, #1)",4.19,"Ordered to hold an abandoned army post, John D...",historical-fiction fiction western historical ...
97898,Alexander McCall Smith,2003.0,The Full Cupboard of Life (No. 1 Ladies' Detec...,4.02,"Precious Ramotswe, of the No.1 Ladies Detectiv...",mystery fiction africa series mysteries crime ...


<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

Se aprecia que muchos corresponden a comics, y que hay alguno de terror. 

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font>  Seleccionar los tres libros más similares a cada uno de los libros en `books_user`. Para ello se puede hacer una sola consulta a la base de datos, pasando directamente los valores de la columna `overview` de las filas correspondientes en `df_goodreads` como una lista. Almacenar el resultado en una variable denominada `results`

In [52]:
overviews_user = df_goodreads.loc[books_user]['overview'].tolist()

results = collection.query(
    query_texts=overviews_user,
    n_results=3
)

results

{'ids': [['59980', '13223349', '106076'],
  ['19030845', '59960', '107029'],
  ['16992', '616318', '105973'],
  ['59952', '1305', '36'],
  ['11734', '11093329', '61264'],
  ['16982', '953260', '52640'],
  ['101911', '22544024', '534255'],
  ['543339', '11768', '10920'],
  ['97898', '7058', '7036'],
  ['141372', '50', '551536'],
  ['107821', '6303733', '25893709'],
  ['2872', '15705011', '17340050'],
  ['10618', '13516846', '33896'],
  ['28351', '469571', '543873']],
 'embeddings': None,
 'documents': [["Lieutenant James Gordon takes up a new post in the crime-ridden and corrupt city of Gotham, while billionaire Bruce Wayne returns to the scene of his parents' deaths, intent on punishing the criminal element.Collects BATMAN #404-407.",
   "Following his ground-breaking, critically acclaimed run on Detective Comics, writer Scott Snyder (American Vampire) alongside artist Greg Capullo (Spawn)\xa0begins a new era of The Dark Knight as with the relaunch of\xa0Batman, as\xa0a part of DC Comi

<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font>  El resultado de la consulta, almacenado en `results`, contiene un JSON en el que los resultados se almacenan para cada libro por separado. Explorar la estructura y crear cuatro listas denominadas `books_ids`, `books_distances`, `book_titles` y `book_authors` con la información correspondiente. En el caso de los identificadores, puesto que se han almacenado como `String`, convertirlos a entero. 

<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
Una forma "limpia" de hacer el ejercicio es utilizar `itertools.chain`.
</div>

In [54]:
import itertools

book_ids = list(map(lambda book_id: int(book_id), itertools.chain(*results['ids'])))

books_distances = list(itertools.chain.from_iterable(results['distances']))

book_titles = list(itertools.chain.from_iterable(
    [ [meta['title'] for meta in group] for group in results['metadatas'] ]
))

book_authors = list(itertools.chain.from_iterable(
    [ [meta['author'] for meta in group] for group in results['metadatas'] ]
))

<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> A partir de las cuatro listas anteriores, crear un `DataFrame` denominado `results_user`, con cuatro columnas denominadas `book_id`, `authors`, `title` y `distance`. Eliminar duplicados y los libros que ya han sido leídos por el usuario (almacenados en `book_user`). Ordenar los resultados por valor creciente de  `distance`.

In [57]:
results_user = pd.DataFrame({
    'book_id': book_ids,
    'authors': book_authors,
    'title': book_titles,
    'distance': books_distances
})

results_user = results_user.drop_duplicates(subset='book_id')

results_user = results_user[~results_user['book_id'].isin(books_user)]

results_user = results_user.sort_values(by='distance')

results_user.head(10)


,book_id,authors,title,distance
25,7058,Alexander McCall Smith,The Good Husband of Zebra Drive (No. 1 Ladies'...,0.417356
26,7036,Alexander McCall Smith,The Kalahari Typing School for Men (No. 1 Ladi...,0.428726
4,59960,"Frank Miller, Klaus Janson, Lynn Varley",Batman: The Dark Knight Returns (The Dark Knig...,0.496824
34,15705011,Tracy Chevalier,The Last Runaway,0.771933
5,107029,"Bill Finger, Gardner F. Fox, Bob Kane, Jerry R...","The Batman Chronicles, Vol. 1",0.779259
31,6303733,"Richard Russo, Arthur Morey",That Old Cape Magic,0.786344
1,13223349,"Scott Snyder, Greg Capullo, Jonathan Glapion","Batman, Volume 1: The Court of Owls",0.812744
2,106076,"Jeph Loeb, Tim Sale",Batman: Dark Victory,0.845644
19,22544024,Susan Meissner,Secrets of a Charmed Life,0.858884
10,1305,Steven Pressfield,Gates of Fire: An Epic Novel of the Battle of ...,0.900997


<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>


---

<a id="section43"></a> 
### <font color="#00586D"> Priorización de resultados y recomendación</font>

Llegados a este punto, se han obtenido los libros más similares a los leídos y valorados positivamente por el usuario. Una posibilidad a la hora de priorizar los resultados consistiría en utilizar la información del `DataFrame` `df_goodreads` para priorizar los libros. En concreto, la columna `average_rating` contiene la valoración media de cada libro en la plataforma, y puede ser utilizada para priorizar. 

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i> </font> Obtener los 10 libros de `df_goodreads` con mayor `average_rating` de entre los almacenados en (`results_user`).  Mostrar solamente las columnas `title` y `author`. Almacenar el resultado en un `DataFrame` denominado `recomendation`.

In [60]:
results_user['book_id'] = results_user['book_id'].astype(int)
results_user['average_rating'] = results_user['book_id'].map(df_goodreads['average_rating'])
recommendation = results_user.sort_values('average_rating', ascending=False).head(10)[['title', 'authors']]
recommendation


,title,authors
11,The Lord of the Rings: Weapons and Warfare,"Chris Smith, Christopher Lee, Richard Taylor"
10,Gates of Fire: An Epic Novel of the Battle of ...,Steven Pressfield
35,"Losing Hope (Hopeless, #2)",Colleen Hoover
1,"Batman, Volume 1: The Court of Owls","Scott Snyder, Greg Capullo, Jonathan Glapion"
4,Batman: The Dark Knight Returns (The Dark Knig...,"Frank Miller, Klaus Janson, Lynn Varley"
19,Secrets of a Charmed Life,Susan Meissner
7,"Justice, Volume 1","Jim Krueger, Alex Ross, Doug Braithwaite"
17,Weaveworld,Clive Barker
2,Batman: Dark Victory,"Jeph Loeb, Tim Sale"
25,The Good Husband of Zebra Drive (No. 1 Ladies'...,Alexander McCall Smith


<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

Puede observarse que, efectivamente, los títulos recomendados guardan mucha relación con los títulos valorados positivamente por el usuario (son casi todos comics). 

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>


---

<a id="section5"></a> 
##  <font color="#00586D"> 5. Sistema híbrido de recomendación  </font>
<br>

<br> En el apartado anterior se ha utilizado la valoración media de cada libro para priorizar los vecinos más cercanos. Existe otra posibilidad, que consite en utilizar las valoraciones de los libros obtenidas mediante filtrado colaborativo. Esto constituye un sistema híbrido, ya que se mezclan las dos aproximaciones. Por una parte, se seleccionan libros basados en contenido, y de ellos, se muestra el conjunto para el que se predice una mayor puntuación por parte del usuario.
<br>

No existe mucho material disponible en relación a este tipo de modelos. Tampoco una librería de referencia, iba cobrando popularidad [surprise](http://surpriselib.com/) aunque las últimas contribuciones a GitHub se hicieron hace dos años. En primer lugar, se utilizará esta librería para obtener los scores mediante filtrado colaborativo. En concreto, se utilizará el algoritmo SVD (visto en clase). 

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> El autor principal de la librería describe en en una serie de artículos [enlace](http://nicolas-hug.com/blog/) como funciona la versión básica de este algoritmo. La lectura es ***muy recomendable***. 
</div>

En la siguiente celda, se cargan los votos  en la estructura `ratings_data` (usada por `surprise`) y se aprende un modelo SVD.

In [ ]:
#!pip install surprise


In [61]:
from surprise import SVD, Dataset, Reader
from surprise.model_selection.split import train_test_split

reader = Reader()
ratings_data = Dataset.load_from_df(df_ratings, reader).build_full_trainset()
svd = SVD(n_factors=10)
svd.fit(ratings_data);

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i> </font> 
Crear una columna denominada `score` en `results_user` con la puntuación que se predice para libro (el id del usuario se almacenó previamente en `test_user`).

In [62]:
results_user['score'] = results_user['book_id'].apply(lambda book_id: svd.predict(test_user, book_id).est)
results_user


,book_id,authors,title,distance,average_rating,score
25,7058,Alexander McCall Smith,The Good Husband of Zebra Drive (No. 1 Ladies'...,0.417356,4.06,3.971921
26,7036,Alexander McCall Smith,The Kalahari Typing School for Men (No. 1 Ladi...,0.428726,3.99,3.974358
4,59960,"Frank Miller, Klaus Janson, Lynn Varley",Batman: The Dark Knight Returns (The Dark Knig...,0.496824,4.25,4.339786
34,15705011,Tracy Chevalier,The Last Runaway,0.771933,3.78,3.833591
5,107029,"Bill Finger, Gardner F. Fox, Bob Kane, Jerry R...","The Batman Chronicles, Vol. 1",0.779259,3.99,3.906528
31,6303733,"Richard Russo, Arthur Morey",That Old Cape Magic,0.786344,3.30,3.596056
1,13223349,"Scott Snyder, Greg Capullo, Jonathan Glapion","Batman, Volume 1: The Court of Owls",0.812744,4.30,4.282216
2,106076,"Jeph Loeb, Tim Sale",Batman: Dark Victory,0.845644,4.11,4.057657
19,22544024,Susan Meissner,Secrets of a Charmed Life,0.858884,4.17,4.065316
10,1305,Steven Pressfield,Gates of Fire: An Epic Novel of the Battle of ...,0.900997,4.40,4.354093


<div><font size=3 color=#00586D> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

<font color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i> </font> Almacenar los libros con los diez mayores `score` en un `DataFrame` denominado `recommendation`. Mostrarlo.

In [63]:
recommendation = results_user.sort_values('score', ascending=False).head(10)
recommendation


,book_id,authors,title,distance,average_rating,score
11,36,"Chris Smith, Christopher Lee, Richard Taylor",The Lord of the Rings: Weapons and Warfare,1.060990,4.53,4.424137
10,1305,Steven Pressfield,Gates of Fire: An Epic Novel of the Battle of ...,0.900997,4.40,4.354093
4,59960,"Frank Miller, Klaus Janson, Lynn Varley",Batman: The Dark Knight Returns (The Dark Knig...,0.496824,4.25,4.339786
1,13223349,"Scott Snyder, Greg Capullo, Jonathan Glapion","Batman, Volume 1: The Court of Owls",0.812744,4.30,4.282216
35,17340050,Colleen Hoover,"Losing Hope (Hopeless, #2)",0.955991,4.39,4.115711
17,52640,Clive Barker,Weaveworld,1.030487,4.13,4.108689
7,616318,"Jim Krueger, Alex Ross, Doug Braithwaite","Justice, Volume 1",0.988655,4.14,4.067031
19,22544024,Susan Meissner,Secrets of a Charmed Life,0.858884,4.17,4.065316
2,106076,"Jeph Loeb, Tim Sale",Batman: Dark Victory,0.845644,4.11,4.057657
20,534255,Lucy Grealy,Autobiography of a Face,0.902115,3.96,4.029812


Igualmente, se ven algunos comics, aunque el resultado es ligeramente distinto al anterior.

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#00586D"></i> </font></div>